# Training — ML-FiLM surrogate (finite depth)

Depth-conditioned U-Net for the nonlinear four-wave interaction `S_nl`. In addition to the spectrum, the network is conditioned on water depth through a frequency-resolved group-velocity vector plus `k_p d` and depth (FiLM modulation).

Trained on finite-depth WRT labels (shallow 1-10 m plus a subsample of deeper cases).

Pipeline: load data -> preprocess/normalize -> build condition vector -> define FiLM model -> train.

## 1. Load data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SHALLOW_DIR = "data_train_shallow_1_10/"  # depths 1-10 m
DEEP_DIR    = "data_train_shallow/"       # broad depth range
DEEP_FRAC   = 0.3                          # random subsample of the deep set
DEEP_SEED   = 42

shallow_path = SHALLOW_DIR  # depths 1-10 m
deep_path    = DEEP_DIR     # broad depth range (subsampled)

DEEP_FRAC = 0.3
DEEP_SEED = 42

efth_shallow  = pd.read_csv(f'{shallow_path}efth_d20_1_10m.csv',  header=None)
snl_shallow   = pd.read_csv(f'{shallow_path}snl_d20_1_10m.csv',   header=None)
depth_shallow = pd.read_csv(f'{shallow_path}depth_d20_1_10m.csv', header=None)

efth_deep_full  = pd.read_csv(f'{deep_path}efth_d20_1_999m.csv',  header=None)
snl_deep_full   = pd.read_csv(f'{deep_path}snl_d20_1_999m.csv',   header=None)
depth_deep_full = pd.read_csv(f'{deep_path}depth_d20_1_999m.csv', header=None)

n_deep_full = len(efth_deep_full)
n_deep_keep = int(round(n_deep_full * DEEP_FRAC))
_rng_deep = np.random.default_rng(DEEP_SEED)
deep_idx = np.sort(_rng_deep.choice(n_deep_full, size=n_deep_keep, replace=False))

efth_deep  = efth_deep_full.iloc[deep_idx].reset_index(drop=True)
snl_deep   = snl_deep_full.iloc[deep_idx].reset_index(drop=True)
depth_deep = depth_deep_full.iloc[deep_idx].reset_index(drop=True)

print(f"shallow_1_10 samples: {len(efth_shallow)}")
print(f"deep samples (30% of {n_deep_full}): {len(efth_deep)}")

data_input = pd.concat([efth_shallow,  efth_deep],  ignore_index=True)
data_snl   = pd.concat([snl_shallow,   snl_deep],   ignore_index=True)
data_depth = pd.concat([depth_shallow, depth_deep], ignore_index=True)

print(f"combined samples: {len(data_input)}")

x_train_s = np.reshape(data_input, (data_input.shape[0], 24, 40, 1))
y_train_s = np.reshape(data_snl, (data_snl.shape[0], 24, 40, 1))
x_train_s=x_train_s[:, :, :40, :]
y_train_s=y_train_s[:, :, :40, :]
print(x_train_s.shape)
x = np.asarray(x_train_s)
y = np.asarray(y_train_s)
depth = np.asarray(data_depth).reshape(-1)

tol = 1e-18
min_tail_len = 5
min_zero_dirs = 24

spec = x[..., 0]
N, ndirs, nf = spec.shape

is_zero = np.abs(spec) <= tol
zero_dir_count = np.sum(is_zero, axis=1)
freq_is_zero = zero_dir_count >= min_zero_dirs

bad_mask = np.array([
    any(np.all(freq_is_zero[i, j:]) and np.any(~freq_is_zero[i, :j])
        for j in range(nf - min_tail_len + 1))
    for i in range(N)
])
drop_start = np.array([
    next((j for j in range(nf - min_tail_len + 1)
          if np.all(freq_is_zero[i, j:]) and np.any(~freq_is_zero[i, :j])), -1)
    for i in range(N)
])

print(f"Total samples      : {N}")
print(f"Removed samples    : {bad_mask.sum()}")
print(f"Remaining samples  : {(~bad_mask).sum()}")

if bad_mask.sum() > 0:
    bad_idx = np.where(bad_mask)[0]
    print("\nFirst 5 removed samples:")
    for k, i in enumerate(bad_idx[:5], 1):
        print(f"#{k:02d} idx={i}, depth={depth[i]:.3f} m, zero-tail starts at freq bin {drop_start[i]}")

x_train_s = x_train_s[~bad_mask]
y_train_s = y_train_s[~bad_mask]
data_depth = depth[~bad_mask]

print("\nClean shapes:")
print("x_train_s_clean:", x_train_s.shape)
print("y_train_s_clean:", y_train_s.shape)
print("data_depth_clean:", data_depth.shape)

## 2. Preprocess and normalize

Same input/target normalization as the depth-scaled surrogate, plus a shallow-water amplitude term. Depth metadata (`k_p d`, group velocity) is computed here for the condition vector.

In [ ]:
# CONFIG
# ============================================================
NDIR = 24
NFREQ = 40
G_CONST = 9.81
FREQ0 = 0.03453
FREQ_GROWTH = 1.1
THETA0 = 7.5
DTHETA_DEG = 15.0
EPS = 1e-16

# final global scaling floor
Y_SCALE_EPS = 1e-8

# shallow-water amplitude correction controls
Y_AMP_KDP_REF = 0.75
Y_AMP_KDP_FLOOR = 0.20
Y_AMP_POWER = 3.0
Y_AMP_MAX = 1e3

# target compression
Y_LOG_C = 1.0

# ============================================================
# DATA SPLIT
# ============================================================
depth_all = np.asarray(data_depth).reshape(-1)

x_train, x_vali, y_train, y_vali, depth_train, depth_vali = train_test_split(
    x_train_s,
    y_train_s,
    depth_all,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

x_train = x_train.reshape((-1, NDIR, NFREQ, 1))
x_vali  = x_vali.reshape((-1, NDIR, NFREQ, 1))
y_train = y_train.reshape((-1, NDIR, NFREQ, 1))
y_vali  = y_vali.reshape((-1, NDIR, NFREQ, 1))

x_train_ori = x_train.copy()
x_vali_ori  = x_vali.copy()
y_train_ori = y_train.copy()
y_vali_ori  = y_vali.copy()

print("x_train_ori.shape:", x_train_ori.shape)
print("y_train_ori.shape:", y_train_ori.shape)
print("depth_train.shape:", depth_train.shape)
print("depth_vali.shape :", depth_vali.shape)

# ============================================================
# GRID
# ============================================================
f = np.zeros(NFREQ, dtype=np.float64)
f[0] = FREQ0
for n in range(1, NFREQ):
    f[n] = f[n - 1] * FREQ_GROWTH

theta = np.zeros(NDIR, dtype=np.float64)
theta[0] = THETA0
for n in range(1, NDIR):
    theta[n] = THETA0 + DTHETA_DEG * n

dtheta = np.deg2rad(DTHETA_DEG)
df = np.gradient(f)
omega = 2.0 * np.pi * f

# ============================================================
# PHYSICS HELPERS
# ============================================================
def solve_k(omega_in, depth_in, g=9.81, niter=30):
    omega_in = np.asarray(omega_in, dtype=np.float64)[None, :]
    depth_in = np.asarray(depth_in, dtype=np.float64).reshape(-1, 1)

    k = (omega_in ** 2) / g
    k = np.where(k <= 0.0, 1e-12, k)

    for _ in range(niter):
        kd = k * depth_in
        tanh_kd = np.tanh(kd)
        fval = g * k * tanh_kd - omega_in ** 2
        dfval = g * (tanh_kd + kd * (1.0 - tanh_kd ** 2))
        k = k - fval / np.maximum(dfval, 1e-12)

    return k


def compute_kdp_from_spectrum(x_in, depth_in, f_in, dtheta_in, g=9.81):
    E = x_in[..., 0]
    depth_in = np.asarray(depth_in, dtype=np.float64).reshape(-1)

    E1 = np.sum(E, axis=1) * dtheta_in
    ip = np.argmax(E1, axis=1)
    fp = f_in[ip]

    omega_in = 2.0 * np.pi * f_in
    k = solve_k(omega_in, depth_in, g=g, niter=30)
    kd = k * depth_in[:, None]

    kp = k[np.arange(k.shape[0]), ip]
    kdp = kd[np.arange(kd.shape[0]), ip]

    return kp, kdp, fp, ip


def compute_R_with_limiter(kdp, eps=1e-12, kdp_min=0.5):
    Csh1 = 5.5
    Csh2 = 5.0 / 6.0
    Csh3 = -5.0 / 4.0

    kdp = np.asarray(kdp, dtype=np.float64).reshape(-1)
    kdp_eff = np.maximum(kdp, kdp_min)

    R = (
        1.0
        + (Csh1 / kdp_eff)
        * (1.0 - Csh2 * kdp_eff)
        * np.exp(Csh3 * kdp_eff)
    )
    R = np.maximum(R, eps)
    return R, kdp_eff


def compute_y_amp_from_kdp(
    kdp,
    kdp_ref=Y_AMP_KDP_REF,
    kdp_floor=Y_AMP_KDP_FLOOR,
    power=Y_AMP_POWER,
    amp_max=Y_AMP_MAX
):
    kdp = np.asarray(kdp, dtype=np.float64).reshape(-1)
    kdp_safe = np.maximum(kdp, max(kdp_floor, 1e-12))
    ratio = np.where(kdp_safe < kdp_ref, kdp_ref / kdp_safe, 1.0)
    amp = ratio ** power
    amp = np.minimum(amp, amp_max)
    return amp


# ============================================================
# NORMALIZATION
# ============================================================
def normalize_dataset(x_in, y_in, f_in, R_in, kdp_in=None, y_amp_in=None, eps=1e-16, center_dir=True):
    n_samples = x_in.shape[0]
    n_dir = x_in.shape[1]
    dir_center = n_dir // 2   # target index for the peak direction (= 12 for 24 dirs)

    x_raw_norm = np.zeros_like(x_in, dtype=np.float32)
    y_raw_norm = np.zeros_like(y_in, dtype=np.float32)

    f_indices = np.zeros(n_samples, dtype=np.int32)
    f_values = np.zeros(n_samples, dtype=np.float64)
    F_values = np.zeros(n_samples, dtype=np.float64)
    k_indices = np.zeros(n_samples, dtype=np.int32)
    R_values = np.zeros(n_samples, dtype=np.float64)
    dir_shifts = np.zeros(n_samples, dtype=np.int32)

    f_mat = f_in[None, :]

    if y_amp_in is not None:
        y_amp_arr = np.asarray(y_amp_in, dtype=np.float64).reshape(-1)
    elif kdp_in is not None:
        y_amp_arr = compute_y_amp_from_kdp(kdp_in)
    else:
        y_amp_arr = np.ones(n_samples, dtype=np.float64)

    for i in range(n_samples):
        G = x_in[i, :, :, 0].astype(np.float64)
        Y = y_in[i, :, :, 0].astype(np.float64)

        Q = (f_mat ** 11) * (G ** 3)

        k_idx, f_idx = np.unravel_index(np.argmax(Q), Q.shape)
        f_n = float(f_in[f_idx])
        F_n = float(G[k_idx, f_idx])
        R_n = float(R_in[i])

        F_n_safe = max(F_n, eps)
        f_n_safe = max(f_n, eps)
        R_n_safe = max(R_n, eps)
        A_n_safe = max(float(y_amp_arr[i]), eps)

        x_norm_i = Q / max(float(np.max(Q)), eps)

        y_phys_scaled = Y / (R_n_safe * (F_n_safe ** 3) * (f_n_safe ** 11) * A_n_safe)
        y_norm_i = np.sign(y_phys_scaled) * np.log1p(np.abs(y_phys_scaled) / Y_LOG_C)

        # Center the direction axis so the input peak (k_idx) lands at dir_center.
        # Same shift applied to x and y; undone in denorm_y_sample / denorm_y_batch_current.
        if center_dir:
            shift = int(dir_center - k_idx)
            x_norm_i = np.roll(x_norm_i, shift, axis=0)
            y_norm_i = np.roll(y_norm_i, shift, axis=0)
        else:
            shift = 0

        x_raw_norm[i, :, :, 0] = x_norm_i.astype(np.float32)
        y_raw_norm[i, :, :, 0] = y_norm_i.astype(np.float32)

        f_indices[i] = f_idx
        f_values[i] = f_n
        F_values[i] = F_n
        k_indices[i] = k_idx
        R_values[i] = R_n
        dir_shifts[i] = shift

    return (
        x_raw_norm,
        y_raw_norm,
        f_indices,
        f_values,
        F_values,
        k_indices,
        R_values,
        y_amp_arr,
        dir_shifts,
    )


def denorm_y_sample(
    y_norm_2d,
    idx,
    y_scale,
    R_used,
    F_vals,
    f_vals,
    y_amp_vals=None,
    dir_shifts=None,
    eps=1e-16,
    y_log_c=1.0
):
    y_comp = np.asarray(y_norm_2d, dtype=np.float64) * y_scale
    y_rnorm = np.sign(y_comp) * y_log_c * np.expm1(np.abs(y_comp))

    # undo direction centering (shift applied at norm time)
    if dir_shifts is not None:
        shift = int(dir_shifts[idx])
        if shift != 0:
            y_rnorm = np.roll(y_rnorm, -shift, axis=0)

    R = max(float(R_used[idx]), eps)
    F = max(float(F_vals[idx]), eps)
    f0 = max(float(f_vals[idx]), eps)
    A = max(float(y_amp_vals[idx]), eps) if y_amp_vals is not None else 1.0

    return y_rnorm * (R * (F ** 3) * (f0 ** 11) * A)


# ============================================================
# COMPUTE DEPTH-RELATED METADATA
# ============================================================
kp_train, kdp_train, fp_train_phys, ip_train_phys = compute_kdp_from_spectrum(
    x_train_ori, depth_train, f, dtheta, g=G_CONST
)
kp_vali, kdp_vali, fp_vali_phys, ip_vali_phys = compute_kdp_from_spectrum(
    x_vali_ori, depth_vali, f, dtheta, g=G_CONST
)

R_train, kdp_eff_train = compute_R_with_limiter(kdp_train)
R_vali, kdp_eff_vali = compute_R_with_limiter(kdp_vali)

y_amp_train = compute_y_amp_from_kdp(kdp_train)
y_amp_vali = compute_y_amp_from_kdp(kdp_vali)

print("R_train stats:", np.nanmin(R_train), np.nanmean(R_train), np.nanmax(R_train))
print("R_vali  stats:", np.nanmin(R_vali), np.nanmean(R_vali), np.nanmax(R_vali))
print("y_amp_train stats:", np.nanmin(y_amp_train), np.nanmean(y_amp_train), np.nanmax(y_amp_train))
print("y_amp_vali  stats:", np.nanmin(y_amp_vali), np.nanmean(y_amp_vali), np.nanmax(y_amp_vali))

# ============================================================
# APPLY NORMALIZATION
# ============================================================
(
    x_train_raw_norm,
    y_train_raw_norm,
    f_idx_train,
    f_val_train,
    F_val_train,
    k_idx_train,
    R_used_train,
    y_amp_train_used,
    dir_shift_train,
) = normalize_dataset(
    x_train_ori,
    y_train_ori,
    f,
    R_train,
    kdp_in=kdp_train,
    y_amp_in=y_amp_train,
    eps=EPS
)

(
    x_vali_raw_norm,
    y_vali_raw_norm,
    f_idx_vali,
    f_val_vali,
    F_val_vali,
    k_idx_vali,
    R_used_vali,
    y_amp_vali_used,
    dir_shift_vali,
) = normalize_dataset(
    x_vali_ori,
    y_vali_ori,
    f,
    R_vali,
    kdp_in=kdp_vali,
    y_amp_in=y_amp_vali,
    eps=EPS
)

# ============================================================
# FINAL GLOBAL SCALING
# ============================================================
x_scale = max(float(np.std(x_train_raw_norm[:, :, :, 0])), Y_SCALE_EPS)
y_scale = max(float(np.std(y_train_raw_norm[:, :, :, 0])), Y_SCALE_EPS)

x_train = np.empty_like(x_train_raw_norm, dtype=np.float32)
x_vali  = np.empty_like(x_vali_raw_norm, dtype=np.float32)
y_train = np.empty_like(y_train_raw_norm, dtype=np.float32)
y_vali  = np.empty_like(y_vali_raw_norm, dtype=np.float32)

x_train[:, :, :, 0] = x_train_raw_norm[:, :, :, 0] / x_scale
x_vali[:, :, :, 0]  = x_vali_raw_norm[:, :, :, 0] / x_scale

y_train[:, :, :, 0] = y_train_raw_norm[:, :, :, 0] / y_scale
y_vali[:, :, :, 0]  = y_vali_raw_norm[:, :, :, 0] / y_scale

print("x_scale:", x_scale)
print("y_scale:", y_scale)
print("x_train raw max:", float(np.max(np.abs(x_train_raw_norm[:, :, :, 0]))))
print("y_train raw max:", float(np.max(np.abs(y_train_raw_norm[:, :, :, 0]))))
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

## 3. Condition vector (group velocity + k_p d + depth)

The depth condition fed to the FiLM layers: log-compressed frequency-resolved group velocity concatenated with `k_p d` and depth, z-scored on the training set.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

COND_KPD_MODE   = 'log1p'
COND_DEPTH_MODE = 'log1p'
COND_FEATURE_NAME = 'group_velocity_log1p_plus_kpd_log1p_plus_depth_log1p'

def to_nchw(x):
    if isinstance(x, torch.Tensor):
        t = x.detach()
        if t.ndim == 4 and t.shape[-1] == 1:
            t = t.permute(0, 3, 1, 2).contiguous()
        return t.float().cpu()

    x = np.asarray(x)
    if x.ndim == 4 and x.shape[-1] == 1:
        return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().float()
    if x.ndim == 4 and x.shape[1] == 1:
        return torch.from_numpy(x).contiguous().float()
    raise ValueError(f"Expected (N,H,W,1) or (N,1,H,W), got {x.shape}")


def zscore_train_apply_matrix(x_train_2d, x_vali_2d, eps=1e-6):
    x_train_2d = np.asarray(x_train_2d, dtype=np.float64)
    x_vali_2d = np.asarray(x_vali_2d, dtype=np.float64)
    mu = np.mean(x_train_2d, axis=0)
    sig = np.std(x_train_2d, axis=0)
    sig = np.maximum(sig, eps)
    xtr_z = (x_train_2d - mu[None, :]) / sig[None, :]
    xva_z = (x_vali_2d - mu[None, :]) / sig[None, :]
    return xtr_z, xva_z, mu, sig


def replace_bad_with_train_median_matrix(x_train_2d, x_vali_2d):
    x_train_2d = np.asarray(x_train_2d, dtype=np.float64)
    x_vali_2d = np.asarray(x_vali_2d, dtype=np.float64)
    x_train_2d = np.where(np.isfinite(x_train_2d), x_train_2d, np.nan)
    x_vali_2d = np.where(np.isfinite(x_vali_2d), x_vali_2d, np.nan)
    med = np.nanmedian(x_train_2d, axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    x_train_2d = np.where(np.isnan(x_train_2d), med[None, :], x_train_2d)
    x_vali_2d = np.where(np.isnan(x_vali_2d), med[None, :], x_vali_2d)
    return x_train_2d, x_vali_2d, med


def apply_condition_normalization(x_2d, mu, sig, z_clip=None):
    x_2d = np.asarray(x_2d, dtype=np.float64)
    z = (x_2d - mu[None, :]) / np.maximum(sig[None, :], 1e-6)
    if z_clip is not None:
        z = np.clip(z, -z_clip, z_clip)
    return z


def _dispersion_k_from_w_depth(w, d, g=9.81, niter=60, tol=1e-12):
    w = np.asarray(w, dtype=np.float64)
    d = np.asarray(d, dtype=np.float64)
    k = np.maximum((w ** 2) / g, 1e-12)
    for _ in range(niter):
        kd = k * d
        kd_clip = np.clip(kd, -50.0, 50.0)
        th = np.tanh(kd_clip)
        sech2 = 1.0 / np.cosh(kd_clip) ** 2
        fv = g * k * th - w ** 2
        dfv = g * th + g * k * d * sech2
        step = fv / np.maximum(dfv, 1e-12)
        k_new = np.maximum(k - step, 1e-12)
        if np.max(np.abs(k_new - k)) < tol:
            k = k_new
            break
        k = k_new
    return k


def compute_group_velocity_vector(depth, f, g=9.81):
    depth = np.asarray(depth, dtype=np.float64).reshape(-1)
    f = np.asarray(f, dtype=np.float64).reshape(-1)
    d_safe = np.maximum(depth, 1e-6)[:, None]
    w = (2.0 * np.pi * np.maximum(f, 1e-8))[None, :]
    k = _dispersion_k_from_w_depth(w, d_safe, g=g)
    kd = k * d_safe
    c = w / np.maximum(k, 1e-12)
    n = 0.5 * (1.0 + (2.0 * kd) / np.maximum(np.sinh(2.0 * kd), 1e-12))
    cg = n * c
    return cg


def compute_group_velocity_feature_vector(depth, f, compress='log1p'):
    cg = compute_group_velocity_vector(depth, f)
    cg = np.maximum(cg, 0.0)
    if compress == 'none':
        return cg
    if compress == 'log1p':
        return np.log1p(cg)
    raise ValueError("compress must be 'none' or 'log1p'")


def transform_kpd_feature(kdp_1d, mode='log1p', eps=1e-6):
    kdp_1d = np.asarray(kdp_1d, dtype=np.float64).reshape(-1, 1)
    kdp_safe = np.maximum(kdp_1d, eps)
    if mode == 'none':
        return kdp_safe
    if mode == 'log':
        return np.log(kdp_safe)
    if mode == 'log1p':
        return np.log1p(kdp_safe)
    raise ValueError("mode must be 'none', 'log', or 'log1p'")


def transform_depth_feature(depth_1d, mode='log1p', eps=1e-6):
    depth_1d = np.asarray(depth_1d, dtype=np.float64).reshape(-1, 1)
    depth_safe = np.maximum(depth_1d, eps)
    if mode == 'none':
        return depth_safe
    if mode == 'log':
        return np.log(depth_safe)
    if mode == 'log1p':
        return np.log1p(depth_safe)
    raise ValueError("mode must be 'none', 'log', or 'log1p'")


def compute_condition_feature_matrix(depth, f, kdp_1d, compress='log1p', kdp_mode='log1p', depth_mode='log1p'):
    cg_feat = compute_group_velocity_feature_vector(depth, f, compress=compress)
    cg_feat = np.asarray(cg_feat, dtype=np.float64)
    kdp_feat = transform_kpd_feature(kdp_1d, mode=kdp_mode)
    depth_feat = transform_depth_feature(depth, mode=depth_mode)
    return np.concatenate([cg_feat, kdp_feat, depth_feat], axis=1)

## 4. FiLM model definition

In [ ]:
device = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))
print('device:', device)

USE_CHANNELS_LAST = False
use_amp = False

class CircularPad2D_Torch(nn.Module):
    def __init__(self, pad_dir=1, pad_freq=1):
        super().__init__()
        self.pad_dir = int(pad_dir)
        self.pad_freq = int(pad_freq)

    def forward(self, x):
        pd, pf = self.pad_dir, self.pad_freq
        if pd > 0:
            x = F.pad(x, (0, 0, pd, pd), mode="circular")
        if pf > 0:
            x = F.pad(x, (pf, pf, 0, 0), mode="replicate")
        return x


class SharedConditionEncoder(nn.Module):
    def __init__(self, in_dim=42, hidden=16, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, cond_vec_z):
        return self.net(cond_vec_z)


class BlockAdaptiveModulation(nn.Module):
    """
    FiLM modulation with BOTH per-channel gate and per-frequency gain/bias.
    All projections are zero-initialized -> identity modulation at start.
    """
    def __init__(self, cond_code_dim=32, channels=32, nf=40):
        super().__init__()
        self.to_channel = nn.Linear(cond_code_dim, channels)
        self.to_freq_g = nn.Linear(cond_code_dim, nf)
        self.to_freq_b = nn.Linear(cond_code_dim, nf)

        nn.init.zeros_(self.to_channel.weight)
        nn.init.zeros_(self.to_channel.bias)
        nn.init.zeros_(self.to_freq_g.weight)
        nn.init.zeros_(self.to_freq_g.bias)
        nn.init.zeros_(self.to_freq_b.weight)
        nn.init.zeros_(self.to_freq_b.bias)

    def forward(self, x, cond_code):
        _, _, _, w = x.shape

        ch_gate = self.to_channel(cond_code)[:, :, None, None]
        fg = self.to_freq_g(cond_code)
        fb = self.to_freq_b(cond_code)

        if fg.shape[1] != w:
            fg = F.interpolate(fg.unsqueeze(1), size=w, mode="linear", align_corners=False).squeeze(1)
            fb = F.interpolate(fb.unsqueeze(1), size=w, mode="linear", align_corners=False).squeeze(1)

        fg = fg[:, None, None, :]
        fb = fb[:, None, None, :]

        x = (1.0 + ch_gate) * x
        x = (1.0 + fg) * x + fb
        return x


class DoubleConv_OnePad_Adaptive(nn.Module):
    def __init__(self, in_ch, out_ch, cond_code_dim=32, nf=40, pad_dir=1, pad_freq=1):
        super().__init__()
        self.pad = CircularPad2D_Torch(pad_dir=2 * pad_dir, pad_freq=2 * pad_freq)

        self.c1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=0, bias=True)
        self.m1 = BlockAdaptiveModulation(cond_code_dim=cond_code_dim, channels=out_ch, nf=nf)

        self.c2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=0, bias=True)
        self.m2 = BlockAdaptiveModulation(cond_code_dim=cond_code_dim, channels=out_ch, nf=nf)

    def forward(self, x, cond_code):
        x = self.pad(x)

        x = self.c1(x)
        x = self.m1(x, cond_code)
        x = F.relu(x)

        x = self.c2(x)
        x = self.m2(x, cond_code)
        x = F.relu(x)
        return x


class DownFast_Adaptive(nn.Module):
    def __init__(self, in_ch, out_ch, cond_code_dim=32, nf=40, pad_dir=1, pad_freq=1):
        super().__init__()
        self.conv = DoubleConv_OnePad_Adaptive(
            in_ch, out_ch, cond_code_dim=cond_code_dim, nf=nf,
            pad_dir=pad_dir, pad_freq=pad_freq
        )
        self.pool = nn.MaxPool2d(2)

    def forward(self, x, cond_code):
        feat = self.conv(x, cond_code)
        return feat, self.pool(feat)


class UpFast_Adaptive(nn.Module):
    """
    Conditional up-block (FiLM-modulated):
      nearest upsample -> 1x1 proj -> concat skip -> conditional DoubleConv.
    Same per-channel + per-frequency FiLM as the encoder blocks.
    """
    def __init__(self, in_ch, skip_ch, out_ch, cond_code_dim=32, nf=40, pad_dir=1, pad_freq=1):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1, padding=0, bias=True)
        self.conv = DoubleConv_OnePad_Adaptive(
            out_ch + skip_ch, out_ch,
            cond_code_dim=cond_code_dim, nf=nf,
            pad_dir=pad_dir, pad_freq=pad_freq,
        )

    def forward(self, x, skip, cond_code):
        x = self.up(x)
        x = self.proj(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x, cond_code)


class UNet_GroupVelocityAdaptive_24x40(nn.Module):
    """
    2-down U-Net with FiLM conditioning EVERYWHERE:
      - 2 downsamples (24x40 -> 12x20 -> 6x10).
      - Encoder:    down1 + down2 conditioned (per-channel + per-freq FiLM).
      - Bottleneck: conditioned (per-channel + per-freq FiLM).
      - Decoder:    up1   + up2   conditioned (per-channel + per-freq FiLM).
    Forward signature unchanged: model(x, cond_vec_z).
    """
    def __init__(self, base_ch=16, nf=40, cond_in_dim=None, cond_hidden=16, cond_code_dim=32, pad_dir=1, pad_freq=1):
        super().__init__()
        c1 = base_ch
        c2 = base_ch * 2
        c3 = base_ch * 4

        cond_in_dim = int(nf + 2 if cond_in_dim is None else cond_in_dim)  # cg(40) + kpd(1) + depth(1)
        self.cond_encoder = SharedConditionEncoder(in_dim=cond_in_dim, hidden=cond_hidden, out_dim=cond_code_dim)

        # Encoder (conditioned)
        self.down1 = DownFast_Adaptive(1, c1, cond_code_dim=cond_code_dim, nf=nf, pad_dir=pad_dir, pad_freq=pad_freq)
        self.down2 = DownFast_Adaptive(c1, c2, cond_code_dim=cond_code_dim, nf=nf, pad_dir=pad_dir, pad_freq=pad_freq)

        # Bottleneck (conditioned)
        self.bottleneck = DoubleConv_OnePad_Adaptive(
            c2, c3, cond_code_dim=cond_code_dim, nf=nf,
            pad_dir=pad_dir, pad_freq=pad_freq,
        )

        # Decoder (conditioned)
        self.up1 = UpFast_Adaptive(c3, c2, c2, cond_code_dim=cond_code_dim, nf=nf, pad_dir=pad_dir, pad_freq=pad_freq)
        self.up2 = UpFast_Adaptive(c2, c1, c1, cond_code_dim=cond_code_dim, nf=nf, pad_dir=pad_dir, pad_freq=pad_freq)

        self.out = nn.Conv2d(c1, 1, kernel_size=1, padding=0, bias=True)

    def forward(self, x, cond_vec_z):
        cond_code = self.cond_encoder(cond_vec_z)
        f1, p1 = self.down1(x, cond_code)        # encoder L1, conditioned
        f2, p2 = self.down2(p1, cond_code)       # encoder L2, conditioned
        b      = self.bottleneck(p2, cond_code)  # bottleneck,  conditioned
        u1     = self.up1(b, f2, cond_code)      # decoder L1,  conditioned
        u2     = self.up2(u1, f1, cond_code)     # decoder L2,  conditioned
        return self.out(u2)


ModelCondUNet = UNet_GroupVelocityAdaptive_24x40

## 5. Train and save

Builds the z-scored condition matrix, then trains the depth-conditioned U-Net (Adam + MSE, best-validation checkpoint). Saves the weights and the condition-vector normalization stats (needed at inference).

In [ ]:
COND_UNET_EPOCHS = 100
COND_UNET_BATCH_SIZE = 128
COND_UNET_LR = 1e-3
COND_UNET_SAVE_DIR = '.'

Xtr_cu = to_nchw(x_train)
Ytr_cu = to_nchw(y_train)
Xva_cu = to_nchw(x_vali)
Yva_cu = to_nchw(y_vali)

assert Xtr_cu.shape[1:] == (1, NDIR, NFREQ)
assert Ytr_cu.shape[1:] == (1, NDIR, NFREQ)
assert Xva_cu.shape[1:] == (1, NDIR, NFREQ)
assert Yva_cu.shape[1:] == (1, NDIR, NFREQ)

depth1_cu = np.asarray(depth_train).reshape(-1)
depth2_cu = np.asarray(depth_vali).reshape(-1)

if 'kdp_train' in globals() and 'kdp_vali' in globals():
    kdp1_cu = np.asarray(kdp_train, dtype=np.float64).reshape(-1)
    kdp2_cu = np.asarray(kdp_vali, dtype=np.float64).reshape(-1)
else:
    _, kdp1_cu, _, _ = compute_kdp_from_spectrum(x_train_ori, depth1_cu, f, dtheta, g=9.81)
    _, kdp2_cu, _, _ = compute_kdp_from_spectrum(x_vali_ori, depth2_cu, f, dtheta, g=9.81)

cgvec1_cu = compute_condition_feature_matrix(
    depth1_cu, f, kdp1_cu,
    compress='log1p',
    kdp_mode=COND_KPD_MODE,
    depth_mode=COND_DEPTH_MODE
)
cgvec2_cu = compute_condition_feature_matrix(
    depth2_cu, f, kdp2_cu,
    compress='log1p',
    kdp_mode=COND_KPD_MODE,
    depth_mode=COND_DEPTH_MODE
)

cgvec1_cu, cgvec2_cu, cgvec_med_cu = replace_bad_with_train_median_matrix(cgvec1_cu, cgvec2_cu)
cgvec1_z_cu, cgvec2_z_cu, cgvec_mu_cu, cgvec_sig_cu = zscore_train_apply_matrix(cgvec1_cu, cgvec2_cu, eps=1e-6)
cond_in_dim_cu = int(cgvec1_z_cu.shape[1])

print('\nConditional U-Net condition vector checks')
print('  feature type:', COND_FEATURE_NAME)
print('  train shape:', cgvec1_cu.shape)
print('  vali  shape:', cgvec2_cu.shape)
print('  k_p d train range:', float(np.min(kdp1_cu)), float(np.max(kdp1_cu)))
print('  k_p d vali  range:', float(np.min(kdp2_cu)), float(np.max(kdp2_cu)))
print('  depth train range:', float(np.min(depth1_cu)), float(np.max(depth1_cu)))
print('  depth vali  range:', float(np.min(depth2_cu)), float(np.max(depth2_cu)))
print('  cond input dim:', cond_in_dim_cu)
print('  train norm mean/std:', float(np.mean(cgvec1_z_cu)), float(np.std(cgvec1_z_cu)))
print('  vali  norm mean/std:', float(np.mean(cgvec2_z_cu)), float(np.std(cgvec2_z_cu)))

CONDtr_cu = torch.from_numpy(cgvec1_z_cu.astype(np.float32))
CONDva_cu = torch.from_numpy(cgvec2_z_cu.astype(np.float32))

Xtr_cu = Xtr_cu.to(device)
Ytr_cu = Ytr_cu.to(device)
Xva_cu = Xva_cu.to(device)
Yva_cu = Yva_cu.to(device)
CONDtr_cu = CONDtr_cu.to(device)
CONDva_cu = CONDva_cu.to(device)

if USE_CHANNELS_LAST:
    Xtr_cu = Xtr_cu.contiguous(memory_format=torch.channels_last)
    Ytr_cu = Ytr_cu.contiguous(memory_format=torch.channels_last)
    Xva_cu = Xva_cu.contiguous(memory_format=torch.channels_last)
    Yva_cu = Yva_cu.contiguous(memory_format=torch.channels_last)

N_cu = Xtr_cu.shape[0]
Nv_cu = Xva_cu.shape[0]

base_ch_cond = 32
cond_hidden_cond = 16
cond_code_dim_cond = 64

model_cond_unet = ModelCondUNet(
    base_ch=base_ch_cond,
    nf=NFREQ,
    cond_in_dim=cond_in_dim_cu,
    cond_hidden=cond_hidden_cond,
    cond_code_dim=cond_code_dim_cond,
    pad_dir=1,
    pad_freq=1,
).to(device)

if USE_CHANNELS_LAST:
    model_cond_unet = model_cond_unet.to(memory_format=torch.channels_last)

with torch.no_grad():
    xb_smoke = torch.randn(2, 1, NDIR, NFREQ, device=device)
    cb_smoke = torch.randn(2, cond_in_dim_cu, device=device)
    if USE_CHANNELS_LAST:
        xb_smoke = xb_smoke.contiguous(memory_format=torch.channels_last)
    _ = model_cond_unet(xb_smoke, cb_smoke)

print('Conditional U-Net smoke test passed.')

total_params_cond = sum(p.numel() for p in model_cond_unet.parameters())
trainable_params_cond = sum(p.numel() for p in model_cond_unet.parameters() if p.requires_grad)
print(f'Total conditional U-Net parameters: {total_params_cond:,}')
print(f'Trainable conditional U-Net parameters: {trainable_params_cond:,}')

opt_cond = torch.optim.Adam(model_cond_unet.parameters(), lr=COND_UNET_LR)
loss_fn_cond = nn.MSELoss()

best_val_cond = float('inf')
bad_cond = 0
best_state_cond = None
train_losses_cond, val_losses_cond, epochs_ran_cond = [], [], []

for epoch in range(1, COND_UNET_EPOCHS + 1):
    model_cond_unet.train()
    tr_sum_cond = 0.0

    perm_cond = torch.randperm(N_cu, device=device)
    n_steps_cond = (N_cu + COND_UNET_BATCH_SIZE - 1) // COND_UNET_BATCH_SIZE

    for step in range(n_steps_cond):
        sel = perm_cond[step * COND_UNET_BATCH_SIZE:min((step + 1) * COND_UNET_BATCH_SIZE, N_cu)]

        xb = Xtr_cu.index_select(0, sel)
        yb = Ytr_cu.index_select(0, sel)
        cb = CONDtr_cu.index_select(0, sel)

        opt_cond.zero_grad(set_to_none=True)

        if use_amp:
            with torch.amp.autocast(device_type='cuda', enabled=True):
                pred = model_cond_unet(xb, cb)
                loss = loss_fn_cond(pred, yb)
            scaler.scale(loss).backward()
            scaler.step(opt_cond)
            scaler.update()
        else:
            pred = model_cond_unet(xb, cb)
            loss = loss_fn_cond(pred, yb)
            loss.backward()
            opt_cond.step()

        tr_sum_cond += loss.item() * xb.size(0)

    tr_loss_cond = tr_sum_cond / N_cu

    model_cond_unet.eval()
    va_sum_cond = 0.0
    with torch.inference_mode():
        n_steps_v_cond = (Nv_cu + COND_UNET_BATCH_SIZE - 1) // COND_UNET_BATCH_SIZE
        for step in range(n_steps_v_cond):
            b0 = step * COND_UNET_BATCH_SIZE
            b1 = min((step + 1) * COND_UNET_BATCH_SIZE, Nv_cu)

            xb = Xva_cu[b0:b1]
            yb = Yva_cu[b0:b1]
            cb = CONDva_cu[b0:b1]

            if use_amp:
                with torch.amp.autocast(device_type='cuda', enabled=True):
                    pred = model_cond_unet(xb, cb)
                    loss = loss_fn_cond(pred, yb)
            else:
                pred = model_cond_unet(xb, cb)
                loss = loss_fn_cond(pred, yb)

            va_sum_cond += loss.item() * xb.size(0)

    va_loss_cond = va_sum_cond / Nv_cu

    train_losses_cond.append(tr_loss_cond)
    val_losses_cond.append(va_loss_cond)
    epochs_ran_cond.append(epoch)

    print(f'epoch {epoch:03d}/{COND_UNET_EPOCHS} - train_loss {tr_loss_cond:.6e} - val_loss {va_loss_cond:.6e}')

    if va_loss_cond < best_val_cond - COND_UNET_MIN_IMPROVE:
        best_val_cond = va_loss_cond
        bad_cond = 0
        best_state_cond = {k: v.detach().cpu().clone() for k, v in model_cond_unet.state_dict().items()}
        print('  improved, saving best conditional U-Net weights')
    else:
        bad_cond += 1
        print(f'  no improvement ({bad_cond}/{COND_UNET_PATIENCE})')
        if bad_cond >= COND_UNET_PATIENCE:
            print('  early stopping')
            break

if best_state_cond is not None:
    model_cond_unet.load_state_dict(best_state_cond)

model_cond_unet.eval()

# ============================================================
# SAVE
# ============================================================
cond_unet_tag = f'cgvecadaptive_unet_2down_full_mse_logcg_kpd_depth_base{base_ch_cond}_wch'

cond_unet_state_dict_path = os.path.join(COND_UNET_SAVE_DIR, f'unet_faster_24x40_{cond_unet_tag}_state_dict.pt')
cond_unet_checkpoint_path = os.path.join(COND_UNET_SAVE_DIR, f'unet_faster_24x40_{cond_unet_tag}_checkpoint.pt')
cond_unet_history_path = os.path.join(COND_UNET_SAVE_DIR, f'unet_faster_24x40_{cond_unet_tag}_history.npz')
cond_unet_cond_norm_path = os.path.join(COND_UNET_SAVE_DIR, f'cond_norm_{cond_unet_tag}.npz')

torch.save(model_cond_unet.state_dict(), cond_unet_state_dict_path)
torch.save(
    {
        'model_state_dict': model_cond_unet.state_dict(),
        'train_losses': train_losses_cond,
        'val_losses': val_losses_cond,
        'epochs': epochs_ran_cond,
        'best_val': best_val_cond,
    },
    cond_unet_checkpoint_path,
)
np.savez(
    cond_unet_history_path,
    train_losses=np.array(train_losses_cond, dtype=np.float64),
    val_losses=np.array(val_losses_cond, dtype=np.float64),
    epochs=np.array(epochs_ran_cond, dtype=np.int32),
    best_val=np.array([best_val_cond], dtype=np.float64),
)
np.savez(
    cond_unet_cond_norm_path,
    cgvec_mu=cgvec_mu_cu,
    cgvec_sig=cgvec_sig_cu,
    cgvec_med=cgvec_med_cu,
    f=np.asarray(f, dtype=np.float64),
    cond_feature_name=np.array([COND_FEATURE_NAME]),
    cond_in_dim=np.array([cond_in_dim_cu], dtype=np.int32),
    kdp_mode=np.array([COND_KPD_MODE]),
    depth_mode=np.array([COND_DEPTH_MODE]),
    base_ch=np.array([base_ch_cond], dtype=np.int32),
    cond_hidden=np.array([cond_hidden_cond], dtype=np.int32),
    cond_code_dim=np.array([cond_code_dim_cond], dtype=np.int32),
)
print(f'saved: {cond_unet_state_dict_path}')
print(f'saved: {cond_unet_checkpoint_path}')
print(f'saved: {cond_unet_history_path}')
print(f'saved: {cond_unet_cond_norm_path}')